# Camera Basics — Understanding OpenCV and Camera Operations

**Purpose:** Learn basic camera operations without complexity.

This notebook will help you:
- Initialize the camera (JetBot or OpenCV fallback)
- Understand how frames are captured
- Learn about BGR vs RGB color formats
- Test camera pan/tilt controls
- Capture and save snapshots

## 1. Setup and Imports

In [ ]:
import sys
import os
import time
import cv2
import numpy as np
from IPython.display import display, Image
import ipywidgets as widgets

# Add project root to path
project_root = '/home/steve/Notebooks/Projects/AI & Robotics Hackathon Berlin/ITQ_HackLab_Team_2'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.hardware.camera import CameraController

print("✓ Imports complete")

## 2. Initialize Camera

The camera controller tries two methods:
1. **JetBot Camera** - Hardware-accelerated CSI camera (on Jetson Nano)
2. **OpenCV Fallback** - Standard USB/CSI camera via OpenCV

It will automatically use whichever works.

In [ ]:
# Initialize camera
camera = CameraController()

if camera.initialize():
    print(f"\n✓ Camera initialized successfully!")
    print(f"  Source: {camera.camera_source}")
    print(f"  Resolution: {camera.width}x{camera.height}")
    print(f"  FPS: {camera.fps}")
else:
    print("\n✗ Camera initialization failed!")
    print("\nTroubleshooting:")
    print("  1. Check camera is connected")
    print("  2. Run: ls /dev/video*")
    print("  3. Try: sudo usermod -a -G video $USER")

## 3. Capture a Single Frame

Let's capture one frame and understand what we get.

In [ ]:
# Capture frame
frame = camera.read()

if frame is not None:
    print("Frame captured successfully!\n")
    print(f"Frame shape: {frame.shape}")
    print(f"  Height: {frame.shape[0]} pixels")
    print(f"  Width: {frame.shape[1]} pixels")
    print(f"  Channels: {frame.shape[2]} (BGR format)")
    print(f"\nData type: {frame.dtype}")
    print(f"Value range: 0-255 (8-bit per channel)")
    
    # Display the frame
    _, buffer = cv2.imencode('.jpg', frame)
    display(Image(data=buffer.tobytes()))
else:
    print("Failed to capture frame!")

## 4. Understanding Color Formats

**BGR vs RGB:**
- OpenCV uses **BGR** (Blue, Green, Red)
- Most other libraries use **RGB** (Red, Green, Blue)
- JetBot camera returns RGB, which we convert to BGR

Let's see the difference:

In [ ]:
frame = camera.read()

if frame is not None:
    # Split into channels
    b, g, r = cv2.split(frame)
    
    print("Channel values at center pixel:")
    h, w = frame.shape[:2]
    center_y, center_x = h // 2, w // 2
    print(f"  Blue:  {frame[center_y, center_x, 0]}")
    print(f"  Green: {frame[center_y, center_x, 1]}")
    print(f"  Red:   {frame[center_y, center_x, 2]}")
    
    # Show individual channels
    print("\nBlue channel:")
    _, buffer = cv2.imencode('.jpg', b)
    display(Image(data=buffer.tobytes()))
    
    print("Green channel:")
    _, buffer = cv2.imencode('.jpg', g)
    display(Image(data=buffer.tobytes()))
    
    print("Red channel:")
    _, buffer = cv2.imencode('.jpg', r)
    display(Image(data=buffer.tobytes()))

## 5. Live Camera Feed

Display a continuously updating camera feed.

In [ ]:
import threading

# Create widgets
image_widget = widgets.Image(format='jpeg', width=640, height=480)
fps_widget = widgets.HTML(value="<b>FPS:</b> 0.0")

running = False

def update_feed():
    global running
    last_time = time.time()
    frame_count = 0
    
    while running:
        frame = camera.read()
        if frame is not None:
            # Encode and display
            _, buffer = cv2.imencode('.jpg', frame)
            image_widget.value = buffer.tobytes()
            
            # Calculate FPS
            frame_count += 1
            if frame_count % 10 == 0:
                current_time = time.time()
                fps = 10 / (current_time - last_time)
                fps_widget.value = f"<b>FPS:</b> {fps:.1f}"
                last_time = current_time
        
        time.sleep(0.033)  # ~30 FPS

def start_feed(b):
    global running
    running = True
    thread = threading.Thread(target=update_feed)
    thread.start()

def stop_feed(b):
    global running
    running = False

# Create buttons
start_button = widgets.Button(description="Start Live Feed", button_style='success')
stop_button = widgets.Button(description="Stop", button_style='danger')

start_button.on_click(start_feed)
stop_button.on_click(stop_feed)

# Display
display(widgets.HBox([start_button, stop_button]))
display(fps_widget)
display(image_widget)

## 6. Camera Pan/Tilt Controls

Test the servo-controlled pan and tilt movements.

In [ ]:
# Center camera first
camera.center()
print("Camera centered")
time.sleep(1)

# Test pan (left/right)
print("\nTesting pan...")
camera.set_pan(-45)  # Pan left
print("  Panned left (-45°)")
time.sleep(1)

camera.set_pan(45)   # Pan right
print("  Panned right (+45°)")
time.sleep(1)

camera.set_pan(0)    # Center
print("  Centered (0°)")
time.sleep(1)

# Test tilt (up/down)
print("\nTesting tilt...")
camera.set_tilt(-30)  # Tilt down
print("  Tilted down (-30°)")
time.sleep(1)

camera.set_tilt(30)   # Tilt up
print("  Tilted up (+30°)")
time.sleep(1)

camera.set_tilt(0)    # Center
print("  Centered (0°)")

print("\n✓ Pan/tilt test complete")

## 7. Interactive Pan/Tilt Control

In [ ]:
# Create sliders
pan_slider = widgets.IntSlider(
    value=0, min=-90, max=90, step=5,
    description='Pan:', continuous_update=False
)
tilt_slider = widgets.IntSlider(
    value=0, min=-60, max=60, step=5,
    description='Tilt:', continuous_update=False
)

def on_pan_change(change):
    camera.set_pan(change['new'])

def on_tilt_change(change):
    camera.set_tilt(change['new'])

pan_slider.observe(on_pan_change, names='value')
tilt_slider.observe(on_tilt_change, names='value')

# Preset buttons
def center_camera(b):
    pan_slider.value = 0
    tilt_slider.value = 0

def look_down(b):
    pan_slider.value = 0
    tilt_slider.value = -30

def look_forward(b):
    pan_slider.value = 0
    tilt_slider.value = 0

center_btn = widgets.Button(description="Center", button_style='info')
down_btn = widgets.Button(description="Look Down", button_style='info')
forward_btn = widgets.Button(description="Look Forward", button_style='info')

center_btn.on_click(center_camera)
down_btn.on_click(look_down)
forward_btn.on_click(look_forward)

display(widgets.VBox([
    pan_slider,
    tilt_slider,
    widgets.HBox([center_btn, down_btn, forward_btn])
]))

## 8. Capture and Save Snapshot

In [ ]:
# Capture and save
frame = camera.read()

if frame is not None:
    # Generate filename with timestamp
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = f"snapshot_{timestamp}.jpg"
    filepath = os.path.join(project_root, 'logs', filename)
    
    # Save
    cv2.imwrite(filepath, frame)
    print(f"✓ Snapshot saved: {filepath}")
    
    # Display
    _, buffer = cv2.imencode('.jpg', frame)
    display(Image(data=buffer.tobytes()))
else:
    print("Failed to capture snapshot")

## 9. Cleanup

Always stop the live feed and release camera resources when done.

In [ ]:
# Stop live feed
running = False
time.sleep(0.5)

# Release camera
camera.release()
print("✓ Camera released")

## Summary

You've learned:
- ✓ How to initialize the camera (JetBot or OpenCV)
- ✓ Frame structure: height × width × channels (BGR)
- ✓ BGR vs RGB color formats
- ✓ Capturing single frames and live feeds
- ✓ Camera pan/tilt servo control
- ✓ Saving snapshots

**Next:** Move to `02_vision_explained.ipynb` to learn how the perception system works!